Json Parsing and Processing

In [1]:
import json
import os
os.makedirs("data/json_files", exist_ok=True)

In [5]:
# Sample nested JSON data
json_data = {
    "company": "Tech Innovators Inc.",
    "employees": [
        {
            "id": 1,
            "name": "Alice Smith",
            "role": "Software Engineer",
            "skills": ["Python", "JavaScript", "Machine Learning"],
            "projects": [
                {"name": "Project Alpha", "status": "Completed"},
                {"name": "Project Beta", "status": "In Progress"}
            ]
        },
        {
            "id": 2,
            "name": "Bob Johnson",
            "role": "Data Scientist",
            "skills": ["R", "SQL", "Data Visualization"],
            "projects": [
                {"name": "Project Gamma", "status": "Completed"},
                {"name": "Project Delta", "status": "In Progress"}
            ]
        }
    ],
    "departments": {
        "Engineering": {
            "head": "Alice Smith",
            "budget": 500000,
            "team_size": 25
        },
        "Data Science": {
            "head": "Bob Johnson",
            "budget": 300000,
            "team_size": 15
        }
    }
}


In [6]:
json_data

{'company': 'Tech Innovators Inc.',
 'employees': [{'id': 1,
   'name': 'Alice Smith',
   'role': 'Software Engineer',
   'skills': ['Python', 'JavaScript', 'Machine Learning'],
   'projects': [{'name': 'Project Alpha', 'status': 'Completed'},
    {'name': 'Project Beta', 'status': 'In Progress'}]},
  {'id': 2,
   'name': 'Bob Johnson',
   'role': 'Data Scientist',
   'skills': ['R', 'SQL', 'Data Visualization'],
   'projects': [{'name': 'Project Gamma', 'status': 'Completed'},
    {'name': 'Project Delta', 'status': 'In Progress'}]}],
 'departments': {'Engineering': {'head': 'Alice Smith',
   'budget': 500000,
   'team_size': 25},
  'Data Science': {'head': 'Bob Johnson', 'budget': 300000, 'team_size': 15}}}

In [7]:
with open("data/json_files/company_data.json", "w") as json_file:
    json.dump(json_data, json_file, indent=4)

In [10]:
# Save json lines format
json1_data = [
    {"timestamp": "2024-01-01T12:00:00Z", "event": "user_signup", "user_id": 123},
    {"timestamp": "2024-01-01T12:05:00Z", "event": "user_login", "user_id": 123, "page": "/home"},
    {"timestamp": "2024-01-01T12:10:00Z", "event": "purchase", "user_id": 123, "amount": 49.99}
]
with open("data/json_files/events.jsonl", "w") as jsonl_file:
    for entry in json1_data:
        jsonl_file.write(json.dumps(entry) + "\n")


JSON Processing Strategies

In [13]:
from langchain_community.document_loaders import JSONLoader
import json

In [18]:
from langchain_community.document_loaders import JSONLoader

employee_loader = JSONLoader(
    file_path="data/json_files/company_data.json",
    jq_schema=".employees[]",
    text_content=False
)

employee_docs = employee_loader.load()

print(f"Loaded {len(employee_docs)} employee documents.")
print(employee_docs[0].page_content[:200])

Loaded 2 employee documents.
{"id": 1, "name": "Alice Smith", "role": "Software Engineer", "skills": ["Python", "JavaScript", "Machine Learning"], "projects": [{"name": "Project Alpha", "status": "Completed"}, {"name": "Project B


In [21]:
# Method 2: Custom JSON processing for complex structures
from typing import List, Dict
from langchain_core.documents import Document

print("\n--- Custom JSON Processing ---")

def process_json_with_custom_logic(file_path: str) -> List[Document]:
    with open(file_path, "r") as json_file:
        data = json.load(json_file)
    
    documents = []
    
    # Process employees
    for emp in data.get("employees", []):
        content = f"Employee ID: {emp['id']}\nName: {emp['name']}\nRole: {emp['role']}\nSkills: {', '.join(emp['skills'])}\nProjects:\n"
        for proj in emp.get("projects", []):
            content += f"  - {proj['name']} (Status: {proj['status']})\n"
        documents.append(Document(page_content=content))
    
    # Process departments
    for dept_name, dept_info in data.get("departments", {}).items():
        content = f"Department: {dept_name}\nHead: {dept_info['head']}\nBudget: ${dept_info['budget']}\nTeam Size: {dept_info['team_size']}"
        documents.append(Document(page_content=content))
    
    return documents


--- Custom JSON Processing ---


In [22]:
process_json_with_custom_logic("data/json_files/company_data.json")

[Document(metadata={}, page_content='Employee ID: 1\nName: Alice Smith\nRole: Software Engineer\nSkills: Python, JavaScript, Machine Learning\nProjects:\n  - Project Alpha (Status: Completed)\n  - Project Beta (Status: In Progress)\n'),
 Document(metadata={}, page_content='Employee ID: 2\nName: Bob Johnson\nRole: Data Scientist\nSkills: R, SQL, Data Visualization\nProjects:\n  - Project Gamma (Status: Completed)\n  - Project Delta (Status: In Progress)\n'),
 Document(metadata={}, page_content='Department: Engineering\nHead: Alice Smith\nBudget: $500000\nTeam Size: 25'),
 Document(metadata={}, page_content='Department: Data Science\nHead: Bob Johnson\nBudget: $300000\nTeam Size: 15')]